# CNN Baseline for CIFAR-10

This notebook builds the first CNN baseline for the research question:

**Can we learn a function that maps a 32x32 RGB image to one of 10 CIFAR-10 classes?**

Current status:
- The CNN architecture has been defined in `SRC/cnn_model.py`
- The CNN has **not** been fully trained yet
- This notebook gives a clean, understandable workflow for training and inspecting that CNN


In [ ]:
%matplotlib inline

from pathlib import Path
from datetime import datetime
import json
import sys

import matplotlib.pyplot as plt
import torch
from torch import nn

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from SRC.cifar10_data import (
    CIFAR10_CLASSES,
    CIFAR10_MEAN,
    CIFAR10_STD,
    get_cifar10_dataloaders,
)
from SRC.cnn_model import SimpleCIFAR10CNN
from SRC.training import train_one_epoch, evaluate


## 1. Load the Data

We use the PyTorch dataloaders built earlier. They already:
- convert images into tensors
- normalize them using the CIFAR-10 channel statistics
- split the original training set into training and validation subsets

For the first CNN run, we keep the setup simple. Data augmentation is optional and can be turned on later as a tuning choice.


In [ ]:
batch_size = 128
validation_split = 0.1
use_augmentation = False
seed = 42
num_workers = 0

train_loader, val_loader, test_loader = get_cifar10_dataloaders(
    batch_size=batch_size,
    validation_split=validation_split,
    augment=use_augmentation,
    seed=seed,
    num_workers=num_workers,
)

images, labels = next(iter(train_loader))
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Batch image shape: {tuple(images.shape)}")
print(f"Batch label shape: {tuple(labels.shape)}")
print(f"Classes: {CIFAR10_CLASSES}")


## 2. Build the CNN

The CNN is designed to learn a hierarchy of visual features.

- **Early layers** can learn edges, color contrasts, and simple textures.
- **Middle layers** can combine those into local patterns such as curves or object parts.
- **Later layers** can represent higher-level class-relevant features.

A key detail is that each convolutional filter spans **all input channels**. So the first filters are `3x3x3`, not just `3x3`.
Each filter produces one feature map, and many filters give many feature maps.


In [ ]:
model = SimpleCIFAR10CNN()
print(model)
model.describe_feature_shapes()


## 3. Why This Architecture Makes Sense

- The `3x3` convolutions are small and efficient, which is a good fit for `32x32` images.
- The depth grows from `32 -> 64 -> 128` feature maps, so the model can learn richer features as it goes deeper.
- `MaxPool2d` reduces spatial size and builds local translation invariance.
- `AdaptiveAvgPool2d(1, 1)` keeps high-level feature strength while reducing the number of classifier parameters.
- The final linear layer maps the learned representation to the 10 class scores.

Important note: until the network is trained, the filters are random. So the CNN is **capable** of learning edges and object parts, but it is not meaningfully detecting them yet.


## 4. Choose the Loss and Training Hyperparameters

We use `CrossEntropyLoss` because this is a multi-class classification problem with one correct class per image.

Initial hyperparameters:
- `epochs = 5`: enough to see learning begin without making the first run too slow on CPU
- `learning_rate = 1e-3`: a stable starting point for Adam
- `weight_decay = 1e-4`: mild regularization to help generalization
- `dropout = 0.4`: reduces overfitting in the classifier head
- `batch_size = 128`: balances speed and gradient stability


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 5
learning_rate = 1e-3
weight_decay = 1e-4
dropout = 0.4

model = SimpleCIFAR10CNN(num_classes=len(CIFAR10_CLASSES), dropout=dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

history = []
best_val_accuracy = 0.0
best_state = None

print(f"Device: {device}")


In [ ]:
for epoch in range(1, epochs + 1):
    train_metrics = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )
    val_metrics = evaluate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )

    record = {
        "epoch": epoch,
        "train_loss": train_metrics.loss,
        "train_accuracy": train_metrics.accuracy,
        "val_loss": val_metrics.loss,
        "val_accuracy": val_metrics.accuracy,
    }
    history.append(record)

    if val_metrics.accuracy > best_val_accuracy:
        best_val_accuracy = val_metrics.accuracy
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    print(
        f"Epoch {epoch}/{epochs} | "
        f"train_loss={train_metrics.loss:.4f} | train_acc={train_metrics.accuracy:.4f} | "
        f"val_loss={val_metrics.loss:.4f} | val_acc={val_metrics.accuracy:.4f}"
    )


In [ ]:
epoch_numbers = [entry["epoch"] for entry in history]
train_loss = [entry["train_loss"] for entry in history]
val_loss = [entry["val_loss"] for entry in history]
train_acc = [entry["train_accuracy"] for entry in history]
val_acc = [entry["val_accuracy"] for entry in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epoch_numbers, train_loss, marker="o", label="Train loss")
axes[0].plot(epoch_numbers, val_loss, marker="o", label="Validation loss")
axes[0].set_title("CNN Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(epoch_numbers, train_acc, marker="o", label="Train accuracy")
axes[1].plot(epoch_numbers, val_acc, marker="o", label="Validation accuracy")
axes[1].set_title("CNN Training Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
)

print(f"Best validation accuracy: {best_val_accuracy:.4f}")
print(f"Test loss: {test_metrics.loss:.4f}")
print(f"Test accuracy: {test_metrics.accuracy:.4f}")


In [ ]:
run_dir = PROJECT_ROOT / "Data" / "cnn_notebook_runs" / datetime.now().strftime("run_%Y%m%d_%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = run_dir / "best_cnn_notebook.pt"
torch.save(model.state_dict(), checkpoint_path)

summary = {
    "model": "SimpleCIFAR10CNN",
    "device": str(device),
    "epochs": epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "dropout": dropout,
    "validation_split": validation_split,
    "augment": use_augmentation,
    "best_val_accuracy": best_val_accuracy,
    "test_loss": test_metrics.loss,
    "test_accuracy": test_metrics.accuracy,
    "history": history,
}

summary_path = run_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Saved checkpoint to: {checkpoint_path}")
print(f"Saved summary to: {summary_path}")


## 5. Inspect a Few Feature Maps

This cell looks at some intermediate activations after training. The point is not to prove that a map corresponds to a human-readable concept every time, but to see that different filters respond differently to the same image.


In [ ]:
def denormalize(image_tensor):
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    return (image_tensor.cpu() * std + mean).clamp(0.0, 1.0)

sample_image, sample_label = test_loader.dataset[0]
logits, activations = model.forward_with_activations(sample_image.unsqueeze(0).to(device), detach=True)
predicted_label = int(logits.argmax(dim=1).item())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, axis in enumerate(axes.flat):
    axis.imshow(activations["conv1_1"][0, index], cmap="viridis")
    axis.set_title(f"conv1_1 map {index}")
    axis.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(denormalize(sample_image).permute(1, 2, 0))
plt.title(f"True: {CIFAR10_CLASSES[sample_label]} | Predicted: {CIFAR10_CLASSES[predicted_label]}")
plt.axis("off")
plt.show()


## 6. Hyperparameters to Tune Next

The most useful CNN hyperparameters to tune are:
- **Learning rate**: affects how fast and how stably the model learns
- **Batch size**: affects gradient smoothness and training speed
- **Epochs**: affects how long the model has to learn
- **Weight decay**: regularizes the model and can reduce overfitting
- **Dropout**: reduces overfitting in the classifier head
- **Data augmentation**: can improve generalization by varying the training inputs
- **Number of filters**: affects the capacity of each convolutional block
- **Number of convolutional blocks**: affects how complex a hierarchy of features the model can learn

A sensible next comparison is:
1. Train this CNN without augmentation
2. Train the same CNN with augmentation
3. Compare both against the MLP baseline
